<a href="https://colab.research.google.com/github/Akshay-a/AI-Agents/blob/main/android_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import os

print("The external link is unavailable. Please upload 'train.jsonl' and 'eval.jsonl' manually:")
uploaded = files.upload()

# Verify files after upload
for f in ['train.jsonl', 'eval.jsonl']:
    if os.path.exists(f):
        print(f"{f} uploaded successfully: {os.path.getsize(f)} bytes")
    else:
        print(f"Warning: {f} not found in current directory.")

The external link is unavailable. Please upload 'train.jsonl' and 'eval.jsonl' manually:


Saving eval.jsonl to eval.jsonl
Saving train.jsonl to train.jsonl
train.jsonl uploaded successfully: 27382626 bytes
eval.jsonl uploaded successfully: 1441511 bytes


In [3]:
from datasets import Dataset, DatasetDict
import json
import os

def load_and_format_jsonl(filename):
    # Search for the file or any Colab-renamed versions (e.g., 'train (1).jsonl')
    possible_names = [filename, filename.replace('.jsonl', ' (1).jsonl'), filename.replace('.jsonl', ' (2).jsonl')]
    path = next((f for f in possible_names if os.path.exists(f) and os.path.getsize(f) > 0), None)

    if not path:
        print(f"Warning: Could not find a non-empty file for {filename}")
        return []

    print(f"Loading data from: {path} ({os.path.getsize(path)} bytes)")
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            try:
                data = json.loads(line)
                # Your file already contains the 'messages' key directly
                if 'messages' in data:
                    rows.append({'messages': data['messages']})
                else:
                    # Fallback for prompt/completion structure
                    p = data.get('prompt', [])
                    c = data.get('completion', [])
                    if isinstance(p, str): p = [{'role': 'user', 'content': p}]
                    if isinstance(c, str): c = [{'role': 'assistant', 'content': c}]
                    rows.append({'messages': p + c})
            except Exception as e:
                print(f"Error parsing line {i+1} in {path}: {e}")
    return rows

train_rows = load_and_format_jsonl('train.jsonl')
eval_rows = load_and_format_jsonl('eval.jsonl')

if train_rows:
    dataset = DatasetDict({
        'train': Dataset.from_list(train_rows),
        'eval': Dataset.from_list(eval_rows) if eval_rows else Dataset.from_list(train_rows[:1]) # Use 1 row of train as eval if missing
    })
    print(f"\nSuccess! Loaded {len(dataset['train'])} training rows.")
else:
    print("\nStill no data found. Please ensure you have uploaded the files and they are not 0 bytes.")

Loading data from: train.jsonl (27382626 bytes)
Loading data from: eval.jsonl (1441511 bytes)

Success! Loaded 5708 training rows.


In [2]:
!pip install -U trl peft accelerate datasets bitsandbytes wandb
!pip uninstall -y transformers
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U accelerate bitsandbytes safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-riwcj2f2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-riwcj2f2
  Resolved htt

In [4]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


In [5]:
from huggingface_hub import login
import wandb

print("Please login to Hugging Face (Required for Gemma 4):")
login() # You need a token from huggingface.co/settings/tokens

print("\nPlease login to Weights & Biases to track loss curves:")
wandb.login()


Please login to Hugging Face (Required for Gemma 4):


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Please login to Weights & Biases to track loss curves:


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: apsingiakshay46 (apsingiakshay46-university-of-sydney) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-1.7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.config.use_cache = False


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [7]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

training_args = SFTConfig(
    output_dir="./android-agent-sft-qwen17b",
    max_length=1536,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    assistant_only_loss=True,
    packing=False,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    report_to="wandb",
    fp16=True,
    bf16=False,
    max_steps=100,  # smoke run first; remove after it works
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    peft_config=peft_config,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/5708 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

In [9]:
def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print({
        "trainable": trainable,
        "total": total,
        "trainable_percent": round(100 * trainable / total, 4),
    })

print_trainable_parameters(trainer.model)

print(torch.cuda.get_device_name(0))
print("VRAM allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
print("VRAM reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))


{'trainable': 17432576, 'total': 1033364480, 'trainable_percent': 1.687}
Tesla T4
VRAM allocated GB: 1.29
VRAM reserved GB: 3.07


In [10]:
def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print({
        "trainable": trainable,
        "total": total,
        "trainable_percent": round(100 * trainable / total, 4),
    })

print_trainable_parameters(trainer.model)

print(torch.cuda.get_device_name(0))
print("VRAM allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
print("VRAM reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))


{'trainable': 17432576, 'total': 1033364480, 'trainable_percent': 1.687}
Tesla T4
VRAM allocated GB: 1.29
VRAM reserved GB: 3.07


In [12]:
trainer.train()


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'